# Stage 1 V3.2: Fix CBSD Regression

## Problème identifié:
- V3.1 a résolu CGM (0% → 81%) et CBB (0.6% → 34%)
- Mais CBSD a régressé de 49% → 2.9%
- Le modèle confond CBSD avec CGM

## Solution V3.2:
1. **Oversampling CBSD 3x** (pas juste 2x)
2. **Continuer l'entraînement** depuis le checkpoint V3.1
3. **Ajuster class weights** pour CBSD

---

In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================

import os
import json
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
from collections import defaultdict, Counter
from datasets import Dataset
from tqdm import tqdm

from unsloth import FastVisionModel

from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090


In [2]:
# ============================================================
# CELL 2: CONFIGURATION V3.2
# ============================================================

# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
DATA_DIR = BASE_DIR / "data" / "stage1"

# Checkpoint V3.1 à charger (pour continuer l'entraînement)
V31_CHECKPOINT = BASE_DIR / "checkpoints" / "stage1_cassava_gemma3n_v3" / "final_model"

# Nouveau checkpoint V3.2
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage1_cassava_gemma3n_v3.2_cbsd_fix"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "unsloth/gemma-3n-E2B-it-unsloth-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)

# Classes
CLASS_NAMES = [
    "Cassava Bacterial Blight (CBB)",
    "Cassava Brown Streak Disease (CBSD)",
    "Cassava Green Mottle (CGM)",
    "Cassava Mosaic Disease (CMD)",
    "Healthy"
]

# ============================================================
# 🔥 OVERSAMPLING RATIOS pour V3.2
# ============================================================
# CBSD était à 2.9% → besoin de BEAUCOUP plus d'échantillons
OVERSAMPLING_RATIOS = {
    "Cassava Bacterial Blight (CBB)": 1,      # Garder tel quel (34% OK)
    "Cassava Brown Streak Disease (CBSD)": 3, # ⚠️ TRIPLER les échantillons!
    "Cassava Green Mottle (CGM)": 1,          # Garder tel quel (81% OK)
    "Cassava Mosaic Disease (CMD)": 1,        # Garder tel quel (89% OK)
    "Healthy": 2                              # Doubler (40% → viser 50%+)
}

print(f"Configuration V3.2 - CBSD Fix")
print(f"{'='*60}")
print(f"\nOversampling ratios:")
for cls, ratio in OVERSAMPLING_RATIOS.items():
    print(f"  {cls}: {ratio}x")
print(f"\nCheckpoint V3.1: {V31_CHECKPOINT}")
print(f"Output V3.2: {CHECKPOINT_DIR}")

Configuration V3.2 - CBSD Fix

Oversampling ratios:
  Cassava Bacterial Blight (CBB): 1x
  Cassava Brown Streak Disease (CBSD): 3x
  Cassava Green Mottle (CGM): 1x
  Cassava Mosaic Disease (CMD): 1x
  Healthy: 2x

Checkpoint V3.1: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_gemma3n_v3/final_model
Output V3.2: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_gemma3n_v3.2_cbsd_fix


In [3]:
# ============================================================
# CELL 3: PROMPT ET EXTRACTION (corrigés)
# ============================================================

CONSTRAINED_PROMPT_TEMPLATE = """Look at this cassava leaf image and identify the disease.

Choose EXACTLY ONE option:
A) Cassava Bacterial Blight (CBB)
B) Cassava Brown Streak Disease (CBSD)  
C) Cassava Green Mottle (CGM)
D) Cassava Mosaic Disease (CMD)
E) Healthy

Answer with the letter and full name only."""

RESPONSE_TEMPLATES = {
    "Cassava Bacterial Blight (CBB)": "A) Cassava Bacterial Blight (CBB)",
    "Cassava Brown Streak Disease (CBSD)": "B) Cassava Brown Streak Disease (CBSD)",
    "Cassava Green Mottle (CGM)": "C) Cassava Green Mottle (CGM)",
    "Cassava Mosaic Disease (CMD)": "D) Cassava Mosaic Disease (CMD)",
    "Healthy": "E) Healthy"
}

LETTER_TO_CLASS = {
    "A": "Cassava Bacterial Blight (CBB)",
    "B": "Cassava Brown Streak Disease (CBSD)",
    "C": "Cassava Green Mottle (CGM)",
    "D": "Cassava Mosaic Disease (CMD)",
    "E": "Healthy"
}

def extract_prediction(response: str) -> str:
    """Extrait la prédiction - VERSION CORRIGÉE."""
    # Nettoyer: garder après "model\n"
    if "model\n" in response:
        response = response.split("model\n")[-1]
    if "model" in response:
        response = response.split("model")[-1]
    
    response = response.strip()
    
    # Chercher "X)" au début
    for letter in ["A", "B", "C", "D", "E"]:
        if response.startswith(f"{letter})") or response.startswith(f"{letter} )"):
            return LETTER_TO_CLASS[letter]
    
    # Fallback: regex
    import re
    match = re.search(r'\b([A-E])\)', response)
    if match:
        return LETTER_TO_CLASS[match.group(1)]
    
    # Fallback: noms de maladies
    response_lower = response.lower()
    if "cbb" in response_lower or "bacterial blight" in response_lower:
        return "Cassava Bacterial Blight (CBB)"
    elif "cbsd" in response_lower or "brown streak" in response_lower:
        return "Cassava Brown Streak Disease (CBSD)"
    elif "cgm" in response_lower or "green mottle" in response_lower:
        return "Cassava Green Mottle (CGM)"
    elif "cmd" in response_lower or "mosaic" in response_lower:
        return "Cassava Mosaic Disease (CMD)"
    elif "healthy" in response_lower:
        return "Healthy"
    
    return "Unknown"

print("✅ Fonctions d'extraction corrigées")

✅ Fonctions d'extraction corrigées


In [4]:
# ============================================================
# CELL 4: CHARGER MODÈLE V3.1 (continuer l'entraînement)
# ============================================================

print("\n" + "="*60)
print("LOADING MODEL FROM V3.1 CHECKPOINT")
print("="*60)

# Option 1: Charger depuis le checkpoint V3.1 (recommandé)
if V31_CHECKPOINT.exists():
    print(f"✅ Checkpoint V3.1 trouvé: {V31_CHECKPOINT}")
    model, tokenizer = FastVisionModel.from_pretrained(
        str(V31_CHECKPOINT),
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )
    print("✅ Modèle V3.1 chargé - on continue l'entraînement!")
else:
    print(f"⚠️ Checkpoint V3.1 non trouvé, chargement du modèle de base")
    model, tokenizer = FastVisionModel.from_pretrained(
        MODEL_NAME,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
    )
    
    # Ajouter LoRA si nouveau modèle
    model = FastVisionModel.get_peft_model(
        model,
        r=8,
        lora_alpha=8,
        lora_dropout=0.0,
        target_modules=["q_proj", "v_proj"],
        use_rslora=False,
        use_gradient_checkpointing="unsloth"
    )

processor = AutoProcessor.from_pretrained(MODEL_NAME)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")


LOADING MODEL FROM V3.1 CHECKPOINT
✅ Checkpoint V3.1 trouvé: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_gemma3n_v3/final_model
==((====))==  Unsloth 2026.1.1: Fast Gemma3N patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Modèle V3.1 chargé - on continue l'entraînement!

Trainable: 2,187,264 / 4,509,703,616 (0.0485%)


In [5]:
# ============================================================
# CELL 5: CHARGER ET OVERSAMPLE LES DONNÉES
# ============================================================

print("\n" + "="*60)
print("LOADING AND OVERSAMPLING DATA")
print("="*60)

def load_json_dataset(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

cassava_train = load_json_dataset(DATA_DIR / "cassava_train_balanced.json")
cassava_val = load_json_dataset(DATA_DIR / "cassava_val.json")
cassava_test = load_json_dataset(DATA_DIR / "cassava_test.json")

print(f"Données chargées:")
print(f"  Train: {len(cassava_train):,}")
print(f"  Val: {len(cassava_val):,}")
print(f"  Test: {len(cassava_test):,}")

# ============================================================
# Fonctions de formatage
# ============================================================

def extract_image_path(text):
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

def get_class_from_response(response: str) -> str:
    for class_name in CLASS_NAMES:
        if class_name in response:
            return class_name
    return "Unknown"

def get_class_label(sample) -> str:
    for turn in sample['conversations']:
        if turn['from'] == 'assistant':
            return get_class_from_response(turn['value'])
    return "Unknown"

def format_sample_constrained(sample):
    conversations = sample['conversations']
    
    image_path = None
    for turn in conversations:
        if '<img>' in turn['value']:
            image_path = extract_image_path(turn['value'])
            break
    
    class_label = get_class_label(sample)
    
    formatted_conversations = [
        {
            "role": "user",
            "content": f"Picture 1: <image>\n{CONSTRAINED_PROMPT_TEMPLATE}"
        },
        {
            "role": "assistant",
            "content": RESPONSE_TEMPLATES.get(class_label, "E) Healthy")
        }
    ]
    
    return {
        "conversations": formatted_conversations,
        "image_path": image_path,
        "id": sample['id'],
        "class_label": class_label,
        "class_idx": CLASS_NAMES.index(class_label) if class_label in CLASS_NAMES else 4
    }

# ============================================================
# Formater les données
# ============================================================

print("\nFormatting data...")
train_formatted_original = [format_sample_constrained(s) for s in tqdm(cassava_train, desc="Train")]
val_formatted = [format_sample_constrained(s) for s in tqdm(cassava_val, desc="Val")]

# Distribution originale
print("\nDistribution ORIGINALE:")
original_dist = Counter(s['class_label'] for s in train_formatted_original)
for cls, count in original_dist.most_common():
    print(f"  {cls}: {count} ({count/len(train_formatted_original)*100:.1f}%)")


LOADING AND OVERSAMPLING DATA
Données chargées:
  Train: 29,276
  Val: 3,612
  Test: 3,619

Formatting data...


Val: 100%|██████████| 3612/3612 [00:00<00:00, 672100.88it/s]


Distribution ORIGINALE:
  Cassava Mosaic Disease (CMD): 12000 (41.0%)
  Cassava Brown Streak Disease (CBSD): 5296 (18.1%)
  Cassava Green Mottle (CGM): 5054 (17.3%)
  Cassava Bacterial Blight (CBB): 3717 (12.7%)
  Healthy: 3209 (11.0%)


In [6]:
# ============================================================
# CELL 6: APPLIQUER L'OVERSAMPLING
# ============================================================

print("\n" + "="*60)
print("APPLYING OVERSAMPLING")
print("="*60)

# Séparer par classe
samples_by_class = defaultdict(list)
for sample in train_formatted_original:
    samples_by_class[sample['class_label']].append(sample)

# Appliquer oversampling
train_formatted_oversampled = []

for class_name, samples in samples_by_class.items():
    ratio = OVERSAMPLING_RATIOS.get(class_name, 1)
    oversampled = samples * ratio
    train_formatted_oversampled.extend(oversampled)
    print(f"  {class_name}: {len(samples)} × {ratio} = {len(oversampled)}")

# Mélanger
random.seed(42)
random.shuffle(train_formatted_oversampled)

print(f"\n✅ Total après oversampling: {len(train_formatted_oversampled):,}")
print(f"   (était {len(train_formatted_original):,})")

# Nouvelle distribution
print("\nNouvelle distribution:")
new_dist = Counter(s['class_label'] for s in train_formatted_oversampled)
for cls, count in new_dist.most_common():
    pct = count / len(train_formatted_oversampled) * 100
    old_pct = original_dist[cls] / len(train_formatted_original) * 100
    change = pct - old_pct
    print(f"  {cls}: {count} ({pct:.1f}%) [{'+' if change > 0 else ''}{change:.1f}%]")

# Créer datasets
train_dataset = Dataset.from_list(train_formatted_oversampled)
eval_dataset = Dataset.from_list(val_formatted)

print(f"\n✅ Train dataset: {len(train_dataset):,}")
print(f"✅ Eval dataset: {len(eval_dataset):,}")


APPLYING OVERSAMPLING
  Cassava Mosaic Disease (CMD): 12000 × 1 = 12000
  Cassava Brown Streak Disease (CBSD): 5296 × 3 = 15888
  Cassava Green Mottle (CGM): 5054 × 1 = 5054
  Cassava Bacterial Blight (CBB): 3717 × 1 = 3717
  Healthy: 3209 × 2 = 6418

✅ Total après oversampling: 43,077
   (était 29,276)

Nouvelle distribution:
  Cassava Brown Streak Disease (CBSD): 15888 (36.9%) [+18.8%]
  Cassava Mosaic Disease (CMD): 12000 (27.9%) [-13.1%]
  Healthy: 6418 (14.9%) [+3.9%]
  Cassava Green Mottle (CGM): 5054 (11.7%) [-5.5%]
  Cassava Bacterial Blight (CBB): 3717 (8.6%) [-4.1%]

✅ Train dataset: 43,077
✅ Eval dataset: 3,612


In [7]:
# ============================================================
# CELL 7: DATA COLLATOR AVEC AUGMENTATION
# ============================================================

try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False

if HAS_ALBUMENTATIONS:
    train_transforms = A.Compose([
        A.RandomResizedCrop(size=(512, 512), scale=(0.85, 1.0), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Transpose(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=10, p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.GaussNoise(var_limit=(5.0, 30.0), p=0.2),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, 
                        min_holes=1, min_height=8, min_width=8, p=0.2),
    ])
    print("✅ Augmentation activée")
else:
    train_transforms = None
    print("⚠️ Albumentations non disponible")

@dataclass
class Gemma3NDataCollator:
    processor: Any
    transforms: Any = None
    is_training: bool = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        batch_images = []
        batch_class_idx = []

        for feature in features:
            image_path = feature['image_path']
            image = None

            if image_path:
                try:
                    if os.path.exists(image_path):
                        image = Image.open(image_path).convert('RGB')
                        
                        if self.is_training and self.transforms is not None:
                            image_np = np.array(image)
                            augmented = self.transforms(image=image_np)
                            image = Image.fromarray(augmented['image'])
                        
                        image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
                    else:
                        image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')
                except Exception as e:
                    image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')

            batch_images.append([image] if image else None)
            batch_class_idx.append(feature.get('class_idx', 4))

            conversations = feature['conversations']
            messages = []
            for turn in conversations:
                role = turn['role']
                content = turn['content']

                if role == 'user' and image is not None and len(messages) == 0:
                    messages.append({
                        "role": role,
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": content.replace('<image>', '').strip()}
                        ]
                    })
                else:
                    messages.append({
                        "role": role,
                        "content": content.replace('<image>', '').strip()
                    })

            batch_messages.append(messages)

        batch_texts_full = []
        batch_texts_user = []

        for messages in batch_messages:
            full_text = self.processor.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            batch_texts_full.append(full_text)

            user_messages = [msg for msg in messages if msg["role"] == "user"]
            user_text = self.processor.tokenizer.apply_chat_template(
                user_messages, tokenize=False, add_generation_prompt=True
            )
            batch_texts_user.append(user_text)

        batch = self.processor(
            text=batch_texts_full,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        labels = batch["input_ids"].clone()

        for i in range(len(batch_texts_user)):
            user_encoding = self.processor(
                text=[batch_texts_user[i]],
                images=[batch_images[i]],
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=512
            )
            user_length = user_encoding["input_ids"].shape[1]

            if user_length < labels[i].shape[0]:
                labels[i, :user_length] = -100
            else:
                labels[i, :-5] = -100

        batch["labels"] = labels
        batch["class_idx"] = torch.tensor(batch_class_idx)

        return batch

train_collator = Gemma3NDataCollator(processor=processor, transforms=train_transforms, is_training=True)
eval_collator = Gemma3NDataCollator(processor=processor, transforms=None, is_training=False)

print("✅ Data collators créés")

✅ Augmentation activée
✅ Data collators créés


/tmp/ipykernel_1220937/3177942960.py:21: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 30.0), p=0.2),
/tmp/ipykernel_1220937/3177942960.py:22: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32,


In [8]:
# ============================================================
# CELL 8: TRAINER SIMPLIFIÉ
# ============================================================

class CassavaTrainer(Trainer):
    """Trainer sans class weighting (oversampling suffit)."""
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs.pop('class_idx', None)
        outputs = model(**inputs)
        loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        return (loss, outputs) if return_outputs else loss

print("✅ CassavaTrainer configuré")

✅ CassavaTrainer configuré


In [9]:
# ============================================================
# CELL 9: CONFIGURATION TRAINING V3.2
# ============================================================

print("\n" + "="*60)
print("CONFIGURING TRAINER V3.2")
print("="*60)

# Configuration pour fine-tuning supplémentaire
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    
    # Batch size
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    
    # ⚠️ CHANGEMENTS V3.2:
    num_train_epochs=2,                # Seulement 2 epochs (on continue V3.1)
    learning_rate=1e-5,                # Plus bas car fine-tuning
    warmup_ratio=0.05,                 # Moins de warmup
    lr_scheduler_type="cosine",
    
    # Régularisation
    weight_decay=0.01,
    max_grad_norm=0.5,
    
    # Logging
    logging_steps=25,
    save_steps=200,
    save_total_limit=3,
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=100,
    
    # Precision
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # Optimizer
    optim="adamw_8bit",
    
    # Early Stopping
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Misc
    seed=42,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=8,
    early_stopping_threshold=0.005
)

trainer = CassavaTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=train_collator,
    processing_class=tokenizer,
    callbacks=[early_stopping],
)

effective_batch = 2 * 8
total_steps = len(train_dataset) // effective_batch * 2

print(f"\nConfiguration:")
print(f"  - Train samples: {len(train_dataset):,} (avec oversampling)")
print(f"  - Effective batch: {effective_batch}")
print(f"  - Total steps: ~{total_steps:,}")
print(f"  - LR: 1e-5 (fine-tuning)")
print(f"  - Epochs: 2")


CONFIGURING TRAINER V3.2

Configuration:
  - Train samples: 43,077 (avec oversampling)
  - Effective batch: 16
  - Total steps: ~5,384
  - LR: 1e-5 (fine-tuning)
  - Epochs: 2


In [10]:
# ============================================================
# CELL 10: ENTRAÎNEMENT V3.2
# ============================================================

print("\n" + "="*60)
print("STARTING TRAINING V3.2 - CBSD FIX")
print("="*60)
print(f"\n🎯 Objectifs:")
print(f"  - CBSD: 2.9% → >30%")
print(f"  - Garder CGM > 60%")
print(f"  - Garder CMD > 80%")
print(f"  - Garder CBB > 25%")
print("="*60 + "\n")

train_result = trainer.train()

print("\n" + "="*60)
print("TRAINING V3.2 COMPLETED")
print("="*60)
print(f"Final loss: {train_result.training_loss:.4f}")
print(f"Total steps: {train_result.global_step}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 43,077 | Num Epochs = 2 | Total steps = 5,386
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 2,187,264 of 5,441,625,536 (0.04% trained)



STARTING TRAINING V3.2 - CBSD FIX

🎯 Objectifs:
  - CBSD: 2.9% → >30%
  - Garder CGM > 60%
  - Garder CMD > 80%
  - Garder CBB > 25%

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.465700,1.000557
200,0.432100,1.021796
300,0.442400,1.019657
400,0.451100,1.024939
500,0.447200,1.038218
600,0.448200,1.018194
700,0.413000,1.022242
800,0.465200,1.021893
900,0.404500,1.022957



TRAINING V3.2 COMPLETED
Final loss: 0.4463
Total steps: 900


In [11]:
# ============================================================
# CELL 11: SAUVEGARDE
# ============================================================

print("\nSaving model...")
final_model_path = CHECKPOINT_DIR / "final_model"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"✅ Model saved to: {final_model_path}")


Saving model...
✅ Model saved to: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_gemma3n_v3.2_cbsd_fix/final_model


In [12]:
# ============================================================
# CELL 12: ÉVALUATION
# ============================================================

print("\n" + "="*60)
print("EVALUATION ON TEST SET")
print("="*60)

# Formatter le test set
test_formatted = [format_sample_constrained(s) for s in tqdm(cassava_test, desc="Formatting test")]

# Mode inference
FastVisionModel.for_inference(model)

def predict_single(sample, max_new_tokens=50):
    image_path = sample['image_path']
    
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except:
        return "Error", ""
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": CONSTRAINED_PROMPT_TEMPLATE}
            ]
        }
    ]
    
    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    predicted_class = extract_prediction(response)
    
    return predicted_class, response

# Quick test
print("\n--- Quick Test (100 samples) ---")
quick_samples = random.sample(test_formatted, 100)
quick_results = []

for sample in tqdm(quick_samples, desc="Quick test"):
    pred, _ = predict_single(sample)
    quick_results.append({
        'gt': sample['class_label'],
        'pred': pred,
        'correct': pred == sample['class_label']
    })

correct = sum(1 for r in quick_results if r['correct'])
print(f"\n📊 Quick Accuracy: {correct}%")

print("\n📈 Par classe:")
for cls in CLASS_NAMES:
    cls_samples = [r for r in quick_results if r['gt'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        acc = cls_correct / len(cls_samples) * 100
        status = "✅" if acc >= 30 else "❌"
        print(f"  {status} {cls}: {acc:.1f}% ({cls_correct}/{len(cls_samples)})")


EVALUATION ON TEST SET


Formatting test: 100%|██████████| 3619/3619 [00:00<00:00, 791035.81it/s]



--- Quick Test (100 samples) ---


Quick test: 100%|██████████| 100/100 [01:19<00:00,  1.25it/s]


📊 Quick Accuracy: 60%

📈 Par classe:
  ✅ Cassava Bacterial Blight (CBB): 80.0% (4/5)
  ❌ Cassava Brown Streak Disease (CBSD): 0.0% (0/21)
  ✅ Cassava Green Mottle (CGM): 87.5% (7/8)
  ✅ Cassava Mosaic Disease (CMD): 82.1% (46/56)
  ✅ Healthy: 30.0% (3/10)


In [ ]:
# ============================================================
# CELL 13: ÉVALUATION COMPLÈTE (optionnel)
# ============================================================

RUN_FULL_EVAL = True

if RUN_FULL_EVAL:
    print("\n" + "="*60)
    print("FULL EVALUATION")
    print("="*60)
    
    all_results = []
    
    for sample in tqdm(test_formatted, desc="Full test"):
        pred, _ = predict_single(sample)
        all_results.append({
            'gt': sample['class_label'],
            'pred': pred,
            'correct': pred == sample['class_label']
        })
    
    total = len(all_results)
    correct = sum(1 for r in all_results if r['correct'])
    
    print(f"\n📊 ACCURACY GLOBALE: {correct/total*100:.2f}% ({correct}/{total})")
    
    print("\n📈 PAR CLASSE:")
    class_metrics = {}
    for cls in CLASS_NAMES:
        cls_samples = [r for r in all_results if r['gt'] == cls]
        if cls_samples:
            cls_correct = sum(1 for r in cls_samples if r['correct'])
            acc = cls_correct / len(cls_samples) * 100
            class_metrics[cls] = acc
            status = "✅" if acc >= 30 else "❌"
            print(f"  {status} {cls}: {acc:.1f}% ({cls_correct}/{len(cls_samples)})")
    
    # Sauvegarder
    results = {
        'version': 'v3.2_cbsd_fix',
        'global_accuracy': correct/total*100,
        'class_metrics': class_metrics,
        'oversampling_ratios': OVERSAMPLING_RATIOS
    }
    
    with open(CHECKPOINT_DIR / "test_results_v3.2.json", 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n✅ Résultats sauvegardés")
    
    # Décision finale
    print("\n" + "="*60)
    print("DÉCISION FINALE V3.2")
    print("="*60)
    
    cbsd_ok = class_metrics.get("Cassava Brown Streak Disease (CBSD)", 0) >= 30
    cgm_ok = class_metrics.get("Cassava Green Mottle (CGM)", 0) >= 30
    cbb_ok = class_metrics.get("Cassava Bacterial Blight (CBB)", 0) >= 25
    cmd_ok = class_metrics.get("Cassava Mosaic Disease (CMD)", 0) >= 70
    
    if cbsd_ok and cgm_ok and cbb_ok and cmd_ok:
        print("✅✅✅ SUCCÈS COMPLET!")
        print("   Toutes les classes >= seuil minimum")
        print("   Prêt pour Stage 2!")
    else:
        print("⚠️ Succès partiel")
        if not cbsd_ok: print(f"   ❌ CBSD < 30%")
        if not cgm_ok: print(f"   ❌ CGM < 30%")
        if not cbb_ok: print(f"   ❌ CBB < 25%")
        if not cmd_ok: print(f"   ❌ CMD < 70%")

In [ ]:
# ============================================================
# CELL 14: COMPARAISON V3.1 vs V3.2
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║              COMPARAISON V3.1 vs V3.2                        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Classe          │  V3.1    │  V3.2 (cible)  │  Objectif    ║
║  ─────────────────────────────────────────────────────────── ║
║  CMD             │  89.4%   │   > 80%        │  ✅ Garder   ║
║  CGM             │  81.4%   │   > 60%        │  ✅ Garder   ║
║  Healthy         │  40.3%   │   > 40%        │  ✅ Garder   ║
║  CBB             │  34.0%   │   > 25%        │  ✅ Garder   ║
║  CBSD            │   2.9%   │   > 30%        │  🎯 CIBLE    ║
║                                                              ║
║  Solution: Oversampling CBSD 3x                              ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")